# v3

In [1]:
"""
ALS Progression Model Pipeline v3.0
Four-Tier Microglial Pruning–Mitochondrial NAD+ Compensation–Autophagy Collapse Continuum

All symptoms and therapeutic changes are derived from network architecture state.

TIER MODEL:
    Tier 1: Hyperactive microglial pruning (complement-bias + random activity-dependent)
    Tier 2: Mitochondrial stress + NAD+ compensation (NDUFA2/NMNAT3/SIRT model)
    Tier 3: Dynamic autophagy/mitophagy collapse (mTOR / aggregate-load dependent)
    Tier 4: Network decompensation (sparsity + corruption + low NAD cross threshold)

    Vicious-cycle feedback at Tier 2 <-> 3 interface:
        low NAD + high aggregates -> boosted pruning, increased corruption, autophagy failure

AMENDMENTS (v3.0 over v2.0):
    1. MitochondrialNADBuffer: explicit Tier 2 energy buffer with per-layer NAD tracking
    2. Dynamic autophagy collapse in ToxicAggregateTracker (Tier 3 vicious cycle)
    3. AugmentedCheungRegimen (aCGR) treatment: multi-tier oral regimen from Paper 4
       - Core (DXM + CYP2D6i + piracetam) -> Tier 1 plasticity rescue
       - NMN -> Tier 2 NAD+ compensation boost
       - NAC + pulsed senolytics (fisetin/D+Q) -> Tier 3 oxidative/SASP/autophagy support
    4. NAD-dependent pruning amplification (vicious-cycle amplifier)
    5. NAD-dependent weight corruption magnitude scaling
    6. NAD-penalised regrowth efficiency
    7. Symptom extensions: nad_depletion_score, autophagy_efficiency_proxy
    8. NFL biomarker now scales with weight instability AND NAD depletion
    9. aCGR scenario arms (early pulsed, maintenance)
   10. All v2.0 features preserved (sub-layers, gradient-guided regrowth,
       stochastic flares, onset subtypes, riluzole comparator, etc.)

Architecture-to-Symptom Mapping (v3.0):
    prefrontal_limbic sparsity             ->  depression_score (PHQ-like 0-48)
    upper_motor sparsity                   ->  spasticity_score (0-4)
    lower_motor sparsity                   ->  weakness_score (0-4)
    neuromuscular_junction sparsity        ->  fatigue_score (0-10)
    bulbar sub-layer sparsity              ->  bulbar_score (0-4)
    respiratory sub-layer sparsity + NMJ   ->  respiratory_risk_pct (0-100)
    fine_motor sub-layer sparsity          ->  fine_motor_score (0-4)
    overall accuracy                       ->  alsfrs_r_proxy (0-48)
    weight instability * NAD penalty       ->  biomarker_nfl_proxy
    total sparsity                         ->  overall_disability_pct (0-100)
    prefrontal sparsity + weight corrupt   ->  cognitive_score (0-30)
    alive motor connections ratio           ->  motor_integrity_pct (0-100)
    motor-layer noise magnitude            ->  excitotoxicity_load
    mean(1 - NAD) across motor layers      ->  nad_depletion_score (0-100)
    effective clearance efficiency          ->  autophagy_efficiency_proxy (0-100)

Architecture-to-Treatment Mapping (v3.0):
    Ketamine:
        NMDA blockade              ->  noise reduction (on sparsity-derived noise)
        Synaptogenesis (BDNF)      ->  gradient-guided mask regrowth
        Anti-inflammatory          ->  reduced per-cycle pruning rate
        Plasticity window          ->  optimizer LR multiplier
        Decay model                ->  half-life exponential on all effects
    Riluzole:
        Glutamate modulation       ->  noise damping in motor layers
        Mild neuroprotection       ->  slight pruning reduction
        No synaptogenesis          ->  no regrowth boost
        No plasticity boost        ->  no LR change
        Steady-state PK            ->  short half-life, continuous dosing
    aCGR (Augmented Cheung Glutamatergic Regimen):
        Core DXM+piracetam         ->  plasticity rescue (regrowth + pruning reduction)
        NMN                        ->  NAD+ buffer replenishment (Tier 2 rescue)
        NAC                        ->  oxidative/noise reduction
        Pulsed senolytics          ->  autophagy clearance boost (Tier 3 rescue)
        Combined multi-tier hit    ->  breaks vicious cycle at Tier 2-3 interface
"""

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import csv
import json
import copy
from collections import defaultdict


# ============================================================
# SECTION 1: NETWORK ARCHITECTURE
# ============================================================

class MotorCircuitNetwork(nn.Module):
    """
    Each layer maps to a biological compartment.
    Sub-layer branching after NMJ into bulbar, respiratory, and
    fine_motor sub-modules enables symptom-specific sparsity readout.
    Noise injection per layer models excitotoxicity.

    v2.0: Output replaced by three sub-circuits (bulbar 128->64,
    respiratory 128->64, fine_motor 128->64) concatenated into
    final_output 192->4.  Enables bulbar-onset vs limb-onset ALS
    via differential pruning of sub-layers.
    """

    LAYER_NAMES = [
        'prefrontal_limbic',
        'upper_motor',
        'lower_motor',
        'neuromuscular_junction',
        'bulbar',
        'respiratory',
        'fine_motor',
        'final_output'
    ]

    def __init__(self, input_dim=2, output_dim=4):
        super().__init__()
        self.prefrontal_limbic = nn.Linear(input_dim, 512)
        self.upper_motor = nn.Linear(512, 512)
        self.lower_motor = nn.Linear(512, 256)
        self.neuromuscular_junction = nn.Linear(256, 128)
        self.bulbar = nn.Linear(128, 64)
        self.respiratory = nn.Linear(128, 64)
        self.fine_motor = nn.Linear(128, 64)
        self.final_output = nn.Linear(192, output_dim)
        self.relu = nn.ReLU()

    def forward(self, x, noise_profile=None):
        if noise_profile is None:
            noise_profile = {}

        h = self.prefrontal_limbic(x)
        if 'prefrontal_limbic' in noise_profile:
            h = h + torch.randn_like(h) * noise_profile['prefrontal_limbic']
        h = self.relu(h)

        h = self.upper_motor(h)
        if 'upper_motor' in noise_profile:
            h = h + torch.randn_like(h) * noise_profile['upper_motor']
        h = self.relu(h)

        h = self.lower_motor(h)
        if 'lower_motor' in noise_profile:
            h = h + torch.randn_like(h) * noise_profile['lower_motor']
        h = self.relu(h)

        h_nmj = self.neuromuscular_junction(h)
        if 'neuromuscular_junction' in noise_profile:
            h_nmj = h_nmj + torch.randn_like(h_nmj) * noise_profile['neuromuscular_junction']
        h_nmj = self.relu(h_nmj)

        h_bulbar = self.bulbar(h_nmj)
        if 'bulbar' in noise_profile:
            h_bulbar = h_bulbar + torch.randn_like(h_bulbar) * noise_profile['bulbar']
        h_bulbar = self.relu(h_bulbar)

        h_resp = self.respiratory(h_nmj)
        if 'respiratory' in noise_profile:
            h_resp = h_resp + torch.randn_like(h_resp) * noise_profile['respiratory']
        h_resp = self.relu(h_resp)

        h_fine = self.fine_motor(h_nmj)
        if 'fine_motor' in noise_profile:
            h_fine = h_fine + torch.randn_like(h_fine) * noise_profile['fine_motor']
        h_fine = self.relu(h_fine)

        h_combined = torch.cat([h_bulbar, h_resp, h_fine], dim=1)
        out = self.final_output(h_combined)
        return out

    def get_layer_params(self):
        return {
            'prefrontal_limbic': (self.prefrontal_limbic.weight,
                                   self.prefrontal_limbic.bias),
            'upper_motor': (self.upper_motor.weight, self.upper_motor.bias),
            'lower_motor': (self.lower_motor.weight, self.lower_motor.bias),
            'neuromuscular_junction': (self.neuromuscular_junction.weight,
                                       self.neuromuscular_junction.bias),
            'bulbar': (self.bulbar.weight, self.bulbar.bias),
            'respiratory': (self.respiratory.weight, self.respiratory.bias),
            'fine_motor': (self.fine_motor.weight, self.fine_motor.bias),
            'final_output': (self.final_output.weight, self.final_output.bias)
        }


# ============================================================
# SECTION 2: MITOCHONDRIAL NAD BUFFER (Tier 2) — NEW v3.0
# ============================================================

class MitochondrialNADBuffer:
    """Tier 2: Pruning-induced mitochondrial stress + NAD+ compensation.

    Biology modelled:
        - NDUFA2 depletion under energy demand (sparsity + excitotoxic noise)
        - NMNAT3/SIRT-like compensatory upregulation (automatic but fragile)
        - Compensation fails when sparsity is high (fewer healthy mitochondria)
        - Low NAD feeds back to pruning (amplifier), corruption (magnitude),
          and regrowth (penalty) — vicious-cycle interface with Tier 3

    All state is per-layer, matching the network architecture.
    NAD levels are bounded [0.0, 1.0] where 1.0 = healthy baseline.
    """

    def __init__(self, layer_names, initial_nad=1.0, depletion_rate=0.015,
                 compensation_strength=0.008, burnout_threshold=0.25):
        self.layer_names = layer_names
        self.nad_levels = {name: initial_nad for name in layer_names}
        self.history = {name: [] for name in layer_names}
        self.depletion_rate = depletion_rate
        self.compensation_strength = compensation_strength
        self.burnout_threshold = burnout_threshold

    def update(self, sparsities, noise_profile, treatment_nad_boost=0.0):
        """Sparsity + excitotoxicity = energy demand -> NAD depletion.
        Treatment NAD boost (NMN from aCGR) replenishes buffer.
        Compensation (NMNAT3-like) is proportional to remaining healthy tissue.
        """
        for name in self.layer_names:
            sp = sparsities.get(name, 0.0)
            noise = noise_profile.get(name, 0.0)
            demand = sp * 1.5 + noise * 2.0
            depletion = self.depletion_rate * demand
            # Partial compensation — fails when sparsity is high (fewer healthy mito)
            compensation = self.compensation_strength * (1.0 - sp)
            # Treatment NAD boost (e.g. NMN from aCGR), scaled per-cycle
            treatment_comp = treatment_nad_boost * 0.1
            new_nad = self.nad_levels[name] - depletion + compensation + treatment_comp
            self.nad_levels[name] = float(np.clip(new_nad, 0.0, 1.0))

    def get_nad_multiplier(self, layer_name):
        """Returns a multiplier [0.4, 1.0] reflecting NAD-dependent efficiency.
        Below burnout threshold: severe penalty (mitochondrial failure).
        Above: mild modulation proportional to NAD level.
        Used by pruning (amplifier), corruption (magnitude), regrowth (penalty).
        """
        nad = self.nad_levels.get(layer_name, 1.0)
        if nad < self.burnout_threshold:
            return 0.4 + (nad / self.burnout_threshold) * 0.6
        return 0.8 + 0.2 * nad

    def get_average_nad(self, layer_subset=None):
        """Average NAD across specified layers (or all)."""
        if layer_subset is None:
            layer_subset = self.layer_names
        vals = [self.nad_levels.get(ln, 1.0) for ln in layer_subset]
        return float(np.mean(vals)) if vals else 1.0

    def record(self):
        for name in self.layer_names:
            self.history[name].append(self.nad_levels[name])


# ============================================================
# SECTION 3: TOXIC AGGREGATE TRACKER (Tier 3) — UPDATED v3.0
# ============================================================

class ToxicAggregateTracker:
    """
    Protein aggregation load per layer.
    Accumulates when pruned connections are not cleared (autophagy failure).
    High load accelerates further pruning, blocks regrowth,
    AND directly corrupts surviving weights (TDP-43 spread model).

    v2.0: corrupt_surviving_weights() perturbs alive weights near
    pruned connections (architectural toxicity).

    v3.0 additions:
        - attempt_clearance now implements dynamic Tier 3 autophagy collapse:
          clearance efficiency degrades with high aggregate load + low NAD
          (mTOR upregulation / mitophagy failure model).
        - corrupt_surviving_weights magnitude scales with NAD penalty
          (2.0 - nad_mult): lower NAD = stronger corruption.
        - Per-layer effective clearance tracked for symptom mapping.
    """

    def __init__(self, layer_names, clearance_rate=0.05,
                 accumulation_rate=0.8, toxicity_threshold=0.3,
                 corruption_spread_radius=5, corruption_magnitude=0.005):
        self.layer_names = layer_names
        self.clearance_rate = clearance_rate
        self.accumulation_rate = accumulation_rate
        self.toxicity_threshold = toxicity_threshold
        self.corruption_spread_radius = corruption_spread_radius
        self.corruption_magnitude = corruption_magnitude
        self.aggregates = {name: 0.0 for name in layer_names}
        self.history = {name: [] for name in layer_names}
        # v3.0: track effective clearance per layer for symptom mapping
        self.last_clearance_efficiency = {name: clearance_rate for name in layer_names}

    def accumulate(self, layer_name, newly_pruned_fraction):
        self.aggregates[layer_name] += newly_pruned_fraction * self.accumulation_rate

    def attempt_clearance(self, autophagy_efficiency=None, nad_buffer=None,
                          aggregates=None):
        """Tier 3: Autophagy efficiency is now dynamic (mTOR/aggregate collapse).

        Vicious cycle at Tier 2-3 interface:
            - High aggregates (>0.4) + low NAD -> further autophagy failure
            - Efficiency collapses as: eff *= (0.6 + 0.4 * nad_mult)
              when aggregate load exceeds collapse threshold
            - This creates a self-reinforcing loop: aggregates block clearance,
              which increases aggregates, which depletes NAD further

        Per-layer calculation: each layer can have different collapse dynamics
        depending on its local aggregate load and NAD level.
        """
        base_eff = (autophagy_efficiency if autophagy_efficiency is not None
                    else self.clearance_rate)

        for name in self.layer_names:
            eff = base_eff
            # Vicious cycle: high aggregates + low NAD -> autophagy collapse
            if nad_buffer is not None and aggregates is not None:
                nad_mult = nad_buffer.get_nad_multiplier(name)
                agg = aggregates.get(name, 0.0)
                if agg > 0.4:
                    # Collapse amplified by NAD depletion
                    eff *= (0.6 + 0.4 * nad_mult)
            cleared = self.aggregates[name] * eff
            self.aggregates[name] = max(0.0, self.aggregates[name] - cleared)
            self.last_clearance_efficiency[name] = eff

    def get_toxicity_multiplier(self, layer_name):
        agg = self.aggregates[layer_name]
        if agg > self.toxicity_threshold:
            return 1.0 + (agg - self.toxicity_threshold) * 2.0
        return 1.0

    def get_regrowth_penalty(self, layer_name):
        agg = self.aggregates[layer_name]
        return max(0.0, 1.0 - agg * 1.5)

    def corrupt_surviving_weights(self, model, masks,
                                  newly_pruned_indices_per_layer,
                                  nad_buffer=None):
        """
        Direct weight corruption: perturb alive weights near pruned
        connections.  Models TDP-43 / C9orf72 aggregate spread to
        neighboring synapses.

        v3.0: Magnitude now scales with NAD penalty:
            magnitude = base * (1 + agg) * (2.0 - nad_mult)
        Lower NAD = stronger corruption (mitochondrial failure
        reduces cellular repair capacity).
        """
        layer_params = model.get_layer_params()
        radius = self.corruption_spread_radius

        for layer_name, pruned_indices in newly_pruned_indices_per_layer.items():
            if pruned_indices is None:
                continue
            if torch.is_tensor(pruned_indices) and pruned_indices.numel() == 0:
                continue
            if not torch.is_tensor(pruned_indices):
                pruned_indices = torch.tensor(pruned_indices, dtype=torch.long)
            if pruned_indices.numel() == 0:
                continue

            weight, _ = layer_params[layer_name]
            mask_w = masks[layer_name]['weight']
            agg = self.aggregates.get(layer_name, 0.0)

            # v3.0: NAD-dependent corruption magnitude
            nad_mult = 1.0
            if nad_buffer is not None:
                nad_mult = nad_buffer.get_nad_multiplier(layer_name)
            magnitude = self.corruption_magnitude * (1.0 + agg) * (2.0 - nad_mult)

            flat_weight = weight.data.flatten()
            flat_mask = mask_w.flatten()
            n = flat_weight.numel()

            offsets = torch.arange(-radius, radius + 1)
            expanded = pruned_indices.unsqueeze(1) + offsets.unsqueeze(0)
            expanded = expanded.flatten().clamp(0, n - 1)
            expanded = torch.unique(expanded)

            alive_in_zone = expanded[flat_mask[expanded] == 1]
            num_targets = alive_in_zone.numel()
            if num_targets > 0:
                flat_weight[alive_in_zone] += (
                    torch.randn(num_targets) * magnitude
                )

    def record(self):
        for name in self.layer_names:
            self.history[name].append(self.aggregates[name])

    def get_state(self):
        return {name: self.aggregates[name] for name in self.layer_names}

    def get_clearance_state(self):
        """v3.0: Return last effective clearance per layer for symptom mapping."""
        return {name: self.last_clearance_efficiency[name]
                for name in self.layer_names}


# ============================================================
# SECTION 4: MICROGLIAL PRUNING ENGINE
# ============================================================

class MicroglialPruningEngine:
    """
    Complement-tagged pruning targets weakest abs(weight) connections.
    Activity-dependent pruning removes connections at random.
    Masks (binary) track alive=1 / pruned=0 per weight element.
    Regrowth flips 0->1 and re-initialises weight from N(0, init_std).

    v2.0 additions:
        - prune_layer returns (fraction, indices) for toxicity spread
        - gradient_guided_regrow: BDNF-style selective unmasking
        - accumulate_gradients: EMA of |grad| per weight position
        - get_weight_instability: std dev of alive weights per layer
    """

    def __init__(self, model, complement_bias=0.7):
        self.model = model
        self.complement_bias = complement_bias
        self.masks = {}
        self.sparsity_history = {}
        self.gradient_accumulator = {}

        layer_params = model.get_layer_params()
        for name, (weight, bias) in layer_params.items():
            self.masks[name] = {
                'weight': torch.ones_like(weight.data),
                'bias': torch.ones_like(bias.data)
            }
            self.sparsity_history[name] = []
            self.gradient_accumulator[name] = torch.zeros_like(weight.data)

    def prune_layer(self, layer_name, rate, toxicity_multiplier=1.0):
        effective_rate = min(rate * toxicity_multiplier, 0.95)
        layer_params = self.model.get_layer_params()
        weight, _ = layer_params[layer_name]
        mask_w = self.masks[layer_name]['weight']

        alive = (mask_w == 1)
        num_alive = alive.sum().item()
        if num_alive == 0:
            return 0.0, torch.tensor([], dtype=torch.long)

        num_to_prune = int(effective_rate * num_alive)
        if num_to_prune == 0:
            return 0.0, torch.tensor([], dtype=torch.long)

        all_pruned = []
        num_complement = int(self.complement_bias * num_to_prune)
        num_random = num_to_prune - num_complement

        if num_complement > 0:
            alive_weights = weight.data.abs().clone()
            alive_weights[~alive] = float('inf')
            flat = alive_weights.flatten()
            _, indices = torch.topk(
                flat, min(num_complement, flat.numel()), largest=False
            )
            valid = flat[indices] < float('inf')
            indices = indices[valid]
            mask_w.flatten()[indices] = 0
            all_pruned.append(indices)

        if num_random > 0:
            alive_after = (mask_w == 1)
            alive_indices = torch.nonzero(
                alive_after.flatten(), as_tuple=False
            ).squeeze(-1)
            if alive_indices.numel() > 0:
                count = min(num_random, alive_indices.numel())
                perm = torch.randperm(alive_indices.numel())[:count]
                selected = alive_indices[perm]
                mask_w.flatten()[selected] = 0
                all_pruned.append(selected)

        weight.data *= mask_w
        newly_pruned_frac = num_to_prune / mask_w.numel()

        if all_pruned:
            pruned_indices = torch.cat(all_pruned)
        else:
            pruned_indices = torch.tensor([], dtype=torch.long)

        return newly_pruned_frac, pruned_indices

    def regrow_layer(self, layer_name, fraction, regrowth_penalty=1.0,
                     init_std=0.03):
        effective_fraction = fraction * regrowth_penalty
        layer_params = self.model.get_layer_params()
        weight, _ = layer_params[layer_name]
        mask_w = self.masks[layer_name]['weight']

        dead = (mask_w == 0)
        num_dead = dead.sum().item()
        if num_dead == 0:
            return 0

        num_regrow = int(effective_fraction * num_dead)
        if num_regrow == 0:
            return 0

        dead_indices = torch.nonzero(
            dead.flatten(), as_tuple=False
        ).squeeze(-1)
        count = min(num_regrow, dead_indices.numel())
        perm = torch.randperm(dead_indices.numel())[:count]
        selected = dead_indices[perm]

        mask_w.flatten()[selected] = 1
        weight.data.flatten()[selected] = torch.randn(count) * init_std
        return count

    def accumulate_gradients(self):
        layer_params = self.model.get_layer_params()
        for name, (weight, _) in layer_params.items():
            if weight.grad is not None:
                self.gradient_accumulator[name] = (
                    0.9 * self.gradient_accumulator[name]
                    + 0.1 * weight.grad.data.abs()
                )

    def gradient_guided_regrow(self, layer_name, fraction,
                               regrowth_penalty=1.0, init_std=0.03):
        effective_fraction = fraction * regrowth_penalty
        layer_params = self.model.get_layer_params()
        weight, _ = layer_params[layer_name]
        mask_w = self.masks[layer_name]['weight']
        grad_acc = self.gradient_accumulator[layer_name]

        dead = (mask_w == 0)
        num_dead = dead.sum().item()
        if num_dead == 0:
            return 0

        num_regrow = int(effective_fraction * num_dead)
        if num_regrow == 0:
            return 0

        alive_count_per_row = mask_w.sum(dim=1).clamp(min=1)
        row_mean_grad = (grad_acc * mask_w).sum(dim=1) / alive_count_per_row

        need_scores = row_mean_grad.unsqueeze(1).expand_as(mask_w)
        need_scores = (need_scores * dead.float()
                       + dead.float() * torch.rand_like(need_scores) * 1e-6)

        flat_scores = need_scores.flatten()
        dead_indices = torch.nonzero(
            dead.flatten(), as_tuple=False
        ).squeeze(-1)
        if dead_indices.numel() == 0:
            return 0

        dead_scores = flat_scores[dead_indices]
        count = min(num_regrow, dead_indices.numel())

        if dead_scores.max() < 1e-5:
            perm = torch.randperm(dead_indices.numel())[:count]
            selected = dead_indices[perm]
        else:
            _, top_k = torch.topk(dead_scores, count, largest=True)
            selected = dead_indices[top_k]

        mask_w.flatten()[selected] = 1
        weight.data.flatten()[selected] = torch.randn(count) * init_std
        return count

    def get_weight_instability(self):
        layer_params = self.model.get_layer_params()
        instabilities = {}
        for name, (weight, _) in layer_params.items():
            mask_w = self.masks[name]['weight']
            alive = (mask_w == 1)
            if alive.sum() == 0:
                instabilities[name] = 0.0
            else:
                instabilities[name] = weight.data[alive].std().item()
        return instabilities

    def apply_all_masks(self):
        layer_params = self.model.get_layer_params()
        for name in self.masks:
            weight, _ = layer_params[name]
            weight.data *= self.masks[name]['weight']

    def get_layer_sparsity(self, layer_name):
        mask_w = self.masks[layer_name]['weight']
        total = mask_w.numel()
        dead = (mask_w == 0).sum().item()
        return dead / total if total > 0 else 0.0

    def get_all_sparsities(self):
        return {name: self.get_layer_sparsity(name) for name in self.masks}

    def get_total_sparsity(self):
        total, dead = 0, 0
        for name in self.masks:
            m = self.masks[name]['weight']
            total += m.numel()
            dead += (m == 0).sum().item()
        return dead / total if total > 0 else 0.0

    def record_sparsities(self):
        for name in self.masks:
            self.sparsity_history[name].append(self.get_layer_sparsity(name))

    def get_alive_counts(self):
        counts = {}
        for name in self.masks:
            m = self.masks[name]['weight']
            counts[name] = (m == 1).sum().item()
        return counts

    def get_total_counts(self):
        counts = {}
        for name in self.masks:
            m = self.masks[name]['weight']
            counts[name] = m.numel()
        return counts


# ============================================================
# SECTION 5: SYMPTOM MAPPER — UPDATED v3.0
# ============================================================

class SymptomMapper:
    """
    Every symptom is a direct function of network architecture state.
    No free parameters disconnected from the model.

    v3.0 additions:
        - nad_depletion_score: mean(1 - NAD) across motor layers * 100
        - autophagy_efficiency_proxy: mean effective clearance * 100
        - biomarker_nfl_proxy now scales with NAD penalty
    """

    def __init__(self):
        self.history = defaultdict(list)

    def compute_symptoms(self, sparsities, accuracy, aggregate_state,
                         total_sparsity, noise_profile, alive_counts,
                         total_counts, treatment_active,
                         weight_instabilities=None, nad_levels=None,
                         autophagy_efficiencies=None):
        symptoms = {}

        pl_sp = sparsities.get('prefrontal_limbic', 0.0)
        um_sp = sparsities.get('upper_motor', 0.0)
        lm_sp = sparsities.get('lower_motor', 0.0)
        nmj_sp = sparsities.get('neuromuscular_junction', 0.0)
        bul_sp = sparsities.get('bulbar', 0.0)
        resp_sp = sparsities.get('respiratory', 0.0)
        fine_sp = sparsities.get('fine_motor', 0.0)
        final_sp = sparsities.get('final_output', 0.0)

        # Derived from prefrontal_limbic sparsity
        symptoms['depression_score'] = pl_sp * 48.0

        # Derived from upper_motor sparsity
        symptoms['spasticity_score'] = um_sp * 4.0

        # Derived from lower_motor sparsity
        symptoms['weakness_score'] = lm_sp * 4.0

        # Derived from NMJ sparsity
        symptoms['fatigue_score'] = nmj_sp * 10.0

        # Derived from bulbar sub-layer sparsity
        symptoms['bulbar_score'] = bul_sp * 4.0

        # Derived from respiratory sub-layer + NMJ
        symptoms['respiratory_risk_pct'] = (
            resp_sp * 0.6 + nmj_sp * 0.4
        ) * 100.0

        # Derived from fine_motor sub-layer sparsity
        symptoms['fine_motor_score'] = fine_sp * 4.0

        # Derived from overall network accuracy
        symptoms['alsfrs_r_proxy'] = accuracy * 48.0 / 100.0

        # Derived from weight instability * NAD penalty (v3.0)
        if weight_instabilities is not None:
            total_instability = sum(weight_instabilities.values())
            nad_penalty = 1.0
            if nad_levels is not None:
                avg_nad = float(np.mean(list(nad_levels.values())))
                nad_penalty = 2.0 - avg_nad  # lower NAD = higher NFL
            symptoms['biomarker_nfl_proxy'] = total_instability * 100.0 * nad_penalty
        else:
            total_agg = sum(aggregate_state.values())
            symptoms['biomarker_nfl_proxy'] = total_agg * 100.0

        # Derived from total sparsity
        symptoms['overall_disability_pct'] = total_sparsity * 100.0

        # Derived from prefrontal sparsity + aggregate + weight corruption
        pl_agg = aggregate_state.get('prefrontal_limbic', 0.0)
        pl_instability = 0.0
        if weight_instabilities is not None:
            pl_instability = weight_instabilities.get('prefrontal_limbic', 0.0)
        cog = ((1.0 - pl_sp) * 30.0
               - pl_agg * 10.0
               - pl_instability * 5.0)
        symptoms['cognitive_score'] = max(0.0, cog)

        # Derived from alive connections in all motor layers
        motor_layers = ['upper_motor', 'lower_motor', 'neuromuscular_junction',
                        'bulbar', 'respiratory', 'fine_motor']
        motor_alive = sum(alive_counts.get(ln, 0) for ln in motor_layers)
        motor_total = sum(total_counts.get(ln, 0) for ln in motor_layers)
        symptoms['motor_integrity_pct'] = (
            motor_alive / motor_total * 100.0 if motor_total > 0 else 0.0
        )

        # Derived from noise magnitude in motor layers
        motor_noise = sum(noise_profile.get(ln, 0.0) for ln in motor_layers)
        symptoms['excitotoxicity_load'] = motor_noise

        # v3.0: NAD depletion score — mean(1 - NAD) across motor layers
        if nad_levels is not None:
            nad_vals = [nad_levels.get(ln, 1.0) for ln in motor_layers]
            symptoms['nad_depletion_score'] = float(np.mean([1.0 - v for v in nad_vals])) * 100.0
        else:
            symptoms['nad_depletion_score'] = 0.0

        # v3.0: Autophagy efficiency proxy — mean effective clearance
        if autophagy_efficiencies is not None:
            eff_vals = list(autophagy_efficiencies.values())
            symptoms['autophagy_efficiency_proxy'] = float(np.mean(eff_vals)) * 100.0
        else:
            symptoms['autophagy_efficiency_proxy'] = 100.0

        # Treatment active flag
        symptoms['treatment_active'] = 1.0 if treatment_active else 0.0

        for key, val in symptoms.items():
            self.history[key].append(val)

        return symptoms


# ============================================================
# SECTION 6: TREATMENT MODULE — UPDATED v3.0
# ============================================================

class KetamineTreatment:
    """
    Each effect maps to a specific network architecture change.
    All effects decay exponentially with configurable half-life.
    v3.0: added nad_boost and autophagy_boost defaults for interface consistency.
    """

    def __init__(self, regrowth_strength=0.6, noise_reduction=0.5,
                 pruning_reduction=0.4, plasticity_boost=2.0,
                 half_life=3, dose=1.0):
        self.regrowth_strength = regrowth_strength * dose
        self.noise_reduction = noise_reduction * dose
        self.pruning_reduction = pruning_reduction * dose
        self.plasticity_boost = plasticity_boost * dose
        self.half_life = half_life
        self.dose = dose
        self.active = False
        self.cycles_since_dose = 0

    def administer(self):
        self.active = True
        self.cycles_since_dose = 0

    def get_decay_factor(self):
        if not self.active:
            return 0.0
        factor = 0.5 ** (self.cycles_since_dose / max(self.half_life, 1e-6))
        if factor < 0.05:
            self.active = False
            return 0.0
        return factor

    def tick(self):
        if self.active:
            self.cycles_since_dose += 1

    def get_current_effects(self, cycle=None):
        decay = self.get_decay_factor()
        return {
            'regrowth_boost': self.regrowth_strength * decay,
            'noise_reduction': self.noise_reduction * decay,
            'pruning_reduction': self.pruning_reduction * decay,
            'plasticity_lr_multiplier': 1.0 + (self.plasticity_boost - 1.0) * decay,
            'selective_regrow_targets': (
                'high_gradient_pruned' if decay > 0.1 else None
            ),
            'active': self.active,
            'decay_factor': decay,
            'treatment_type': 'ketamine',
            'nad_boost': 0.0,
            'autophagy_boost': 1.0,
        }


class RiluzoleTreatment:
    """
    Riluzole comparator: pure glutamate modulation.
    v3.0: added nad_boost and autophagy_boost defaults for interface consistency.
    """

    def __init__(self, noise_reduction=0.3, pruning_reduction=0.15,
                 half_life=2, dose=1.0):
        self.noise_reduction = noise_reduction * dose
        self.pruning_reduction = pruning_reduction * dose
        self.half_life = half_life
        self.dose = dose
        self.active = False
        self.cycles_since_dose = 0

    def administer(self):
        self.active = True
        self.cycles_since_dose = 0

    def get_decay_factor(self):
        if not self.active:
            return 0.0
        factor = 0.5 ** (self.cycles_since_dose / max(self.half_life, 1e-6))
        if factor < 0.05:
            self.active = False
            return 0.0
        return factor

    def tick(self):
        if self.active:
            self.cycles_since_dose += 1

    def get_current_effects(self, cycle=None):
        decay = self.get_decay_factor()
        return {
            'regrowth_boost': 0.0,
            'noise_reduction': self.noise_reduction * decay,
            'pruning_reduction': self.pruning_reduction * decay,
            'plasticity_lr_multiplier': 1.0,
            'selective_regrow_targets': None,
            'active': self.active,
            'decay_factor': decay,
            'treatment_type': 'riluzole',
            'nad_boost': 0.0,
            'autophagy_boost': 1.0,
        }


class AugmentedCheungRegimen:
    """Paper 4: aCGR — multi-tier oral regimen.

    Architecture-to-treatment mapping:
        Core (DXM + CYP2D6i + piracetam) -> Tier 1 plasticity rescue
            - regrowth_boost: gradient-guided mask regrowth (BDNF model)
            - pruning_reduction: anti-inflammatory pruning dampening
            - plasticity_lr_multiplier: enhanced synaptic plasticity

        NMN -> Tier 2 NAD+ compensation boost
            - nad_boost: replenishes mitochondrial NAD buffer
            - breaks energy-depletion arm of vicious cycle

        NAC -> Tier 1/2 oxidative stress reduction
            - noise_reduction: reduces excitotoxic noise (sparsity-derived)

        Pulsed senolytics (fisetin / D+Q) -> Tier 3 autophagy/SASP support
            - autophagy_boost: multiplier on clearance efficiency
            - only active during pulse cycles (intermittent dosing)
            - breaks aggregate-accumulation arm of vicious cycle

    Combined multi-tier hit simultaneously addresses Tier 1 (pruning),
    Tier 2 (NAD), and Tier 3 (autophagy) — predicted to break the
    vicious cycle that single-target agents cannot.
    """

    def __init__(self, core_plasticity=0.65, nad_boost=0.45,
                 nac_oxidative_reduction=0.55,
                 senolytic_clearance_boost=1.8, pulse_cycles=None,
                 half_life=4):
        self.core_plasticity = core_plasticity
        self.nad_boost = nad_boost
        self.nac_reduction = nac_oxidative_reduction
        self.senolytic_boost = senolytic_clearance_boost
        self.pulse_cycles = set(pulse_cycles) if pulse_cycles else set()
        self.half_life = half_life
        self.active = False
        self.cycles_since_dose = 0
        self.is_pulse = False

    def administer(self, is_pulse=False):
        self.active = True
        self.cycles_since_dose = 0
        self.is_pulse = is_pulse

    def get_decay_factor(self):
        if not self.active:
            return 0.0
        factor = 0.5 ** (self.cycles_since_dose / max(self.half_life, 1e-6))
        if factor < 0.05:
            self.active = False
            return 0.0
        return factor

    def tick(self):
        if self.active:
            self.cycles_since_dose += 1

    def get_current_effects(self, cycle=None):
        decay = self.get_decay_factor()
        if decay < 0.05:
            self.active = False
            return {
                'regrowth_boost': 0.0, 'pruning_reduction': 0.0,
                'noise_reduction': 0.0, 'nad_boost': 0.0,
                'oxidative_reduction': 0.0, 'autophagy_boost': 1.0,
                'plasticity_lr_multiplier': 1.0,
                'selective_regrow_targets': None,
                'active': False, 'decay_factor': 0.0,
                'treatment_type': 'aCGR'
            }

        # Continuous core + pulsed senolytics
        is_pulse_cycle = (cycle is not None and cycle in self.pulse_cycles)
        pulse_boost = self.senolytic_boost if (is_pulse_cycle or self.is_pulse) else 1.0

        return {
            'regrowth_boost': self.core_plasticity * decay,
            'pruning_reduction': 0.45 * decay,
            'noise_reduction': self.nac_reduction * decay,
            'nad_boost': self.nad_boost * decay,
            'oxidative_reduction': self.nac_reduction * decay,
            'autophagy_boost': pulse_boost * decay + (1.0 - decay),  # blends to 1.0
            'plasticity_lr_multiplier': 1.0 + 1.2 * decay,
            'selective_regrow_targets': (
                'high_gradient_pruned' if decay > 0.1 else None
            ),
            'active': self.active,
            'decay_factor': decay,
            'treatment_type': 'aCGR'
        }


class TreatmentSchedule:
    """Wraps KetamineTreatment, RiluzoleTreatment, or AugmentedCheungRegimen
    with a dosing calendar.

    v3.0: added aCGR support with pulse_cycles passthrough.
    Uses **kwargs for flexible parameter forwarding to treatment constructors.
    """

    def __init__(self, dose_cycles=None, treatment_type='ketamine', **kwargs):
        self.dose_cycles = set(dose_cycles) if dose_cycles is not None else set()
        self.treatment_type = treatment_type

        if treatment_type == 'ketamine':
            self.treatment = KetamineTreatment(
                regrowth_strength=kwargs.get('regrowth_strength', 0.6),
                noise_reduction=kwargs.get('noise_reduction', 0.5),
                pruning_reduction=kwargs.get('pruning_reduction', 0.4),
                plasticity_boost=kwargs.get('plasticity_boost', 2.0),
                half_life=kwargs.get('half_life', 3),
                dose=kwargs.get('dose', 1.0)
            )
        elif treatment_type == 'riluzole':
            self.treatment = RiluzoleTreatment(
                noise_reduction=kwargs.get('noise_reduction', 0.3),
                pruning_reduction=kwargs.get('pruning_reduction', 0.15),
                half_life=kwargs.get('half_life', 2),
                dose=kwargs.get('dose', 1.0)
            )
        elif treatment_type == 'aCGR':
            self.treatment = AugmentedCheungRegimen(
                core_plasticity=kwargs.get('core_plasticity', 0.65),
                nad_boost=kwargs.get('nad_boost', 0.45),
                nac_oxidative_reduction=kwargs.get('nac_oxidative_reduction', 0.55),
                senolytic_clearance_boost=kwargs.get('senolytic_clearance_boost', 1.8),
                pulse_cycles=kwargs.get('pulse_cycles', None),
                half_life=kwargs.get('half_life', 4)
            )
        else:
            raise ValueError(f"Unknown treatment type: {treatment_type}")

    def check_and_administer(self, cycle):
        if cycle in self.dose_cycles:
            if isinstance(self.treatment, AugmentedCheungRegimen):
                is_pulse = cycle in self.treatment.pulse_cycles
                self.treatment.administer(is_pulse=is_pulse)
            else:
                self.treatment.administer()
            return True
        return False

    def get_effects(self, cycle=None):
        return self.treatment.get_current_effects(cycle=cycle)

    def tick(self):
        self.treatment.tick()


# ============================================================
# SECTION 7: EXCITOTOXICITY ENGINE
# ============================================================

class ExcitotoxicityEngine:
    """
    Noise derived from architectural instability.
    noise_per_layer = (base + progression * cycle)
                      * (1 + sparsity * instability_gain)
                      * vulnerability
                      * (1 - treatment_reduction)
    """

    def __init__(self, base_noise=0.05, progression_rate=0.005,
                 motor_vulnerability=1.5, limbic_vulnerability=1.0,
                 instability_gain=3.0):
        self.base_noise = base_noise
        self.progression_rate = progression_rate
        self.motor_vulnerability = motor_vulnerability
        self.limbic_vulnerability = limbic_vulnerability
        self.instability_gain = instability_gain

    def get_noise_profile(self, cycle, current_sparsities=None,
                          treatment_effects=None):
        noise_reduction = 0.0
        if treatment_effects is not None:
            noise_reduction = treatment_effects.get('noise_reduction', 0.0)

        base = self.base_noise + self.progression_rate * cycle
        reduction_mult = max(0.0, 1.0 - noise_reduction)

        if current_sparsities is None:
            current_sparsities = {}

        vulnerability_map = {
            'prefrontal_limbic': self.limbic_vulnerability,
            'upper_motor': self.motor_vulnerability,
            'lower_motor': self.motor_vulnerability * 1.2,
            'neuromuscular_junction': self.motor_vulnerability * 0.8,
            'bulbar': self.motor_vulnerability * 1.0,
            'respiratory': self.motor_vulnerability * 1.1,
            'fine_motor': self.motor_vulnerability * 0.9,
        }

        profile = {}
        for layer_name, vuln in vulnerability_map.items():
            sp = current_sparsities.get(layer_name, 0.0)
            instability_factor = 1.0 + sp * self.instability_gain
            profile[layer_name] = base * instability_factor * vuln * reduction_mult

        return profile


# ============================================================
# SECTION 8: DATA GENERATOR
# ============================================================

class MotorTaskGenerator:
    """
    Synthetic 4-class task:
        0 = limb_function, 1 = bulbar_function,
        2 = respiratory_function, 3 = fine_motor_function
    """

    def __init__(self, n_samples=2000, seed=42):
        self.n_samples = n_samples
        rng = torch.Generator()
        rng.manual_seed(seed)
        self.rng = rng
        self._generate(seed)

    def _generate(self, seed):
        torch.manual_seed(seed)
        centers = torch.tensor([
            [-1.0, -1.0], [1.0, -1.0],
            [-1.0, 1.0],  [1.0, 1.0]
        ])
        x_list, y_list = [], []
        per_class = self.n_samples // 4
        for c in range(4):
            x_c = centers[c] + torch.randn(per_class, 2) * 0.5
            y_c = torch.full((per_class,), c, dtype=torch.long)
            x_list.append(x_c)
            y_list.append(y_c)

        x = torch.cat(x_list)
        y = torch.cat(y_list)
        perm = torch.randperm(len(x))
        x, y = x[perm], y[perm]

        split = int(0.8 * len(x))
        self.x_train, self.x_test = x[:split], x[split:]
        self.y_train, self.y_test = y[:split], y[split:]

    def get_train(self):
        return self.x_train, self.y_train

    def get_test(self):
        return self.x_test, self.y_test


# ============================================================
# SECTION 9: TRAINING AND EVALUATION
# ============================================================

def train_epoch(model, x, y, optimizer, criterion, pruning_engine,
                noise_profile=None, n_steps=5, batch_size=256):
    model.train()
    losses = []
    for _ in range(n_steps):
        idx = torch.randperm(len(x))[:batch_size]
        xb, yb = x[idx], y[idx]
        out = model(xb, noise_profile=noise_profile)
        loss = criterion(out, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        pruning_engine.apply_all_masks()
        losses.append(loss.item())
    return np.mean(losses)


def evaluate(model, x, y, noise_profile=None):
    model.eval()
    with torch.no_grad():
        out = model(x, noise_profile=noise_profile)
        pred = out.argmax(dim=1)
        acc = (pred == y).float().mean().item() * 100.0
        per_class = {}
        for c in range(4):
            mask = (y == c)
            if mask.sum() > 0:
                per_class[c] = (pred[mask] == y[mask]).float().mean().item() * 100.0
            else:
                per_class[c] = 0.0
    return acc, per_class


# ============================================================
# SECTION 10: DISEASE PROFILES
# ============================================================

class DiseaseProfile:
    """
    v2.0 fields per profile + v3.0 additions:
        - All v2.0 fields preserved (pruning_rates, flare params, etc.)
        - nad_depletion_rate, nad_compensation_strength, nad_burnout_threshold
          (Tier 2 parameters; defaults provided if missing)
    """

    @staticmethod
    def als_profile():
        return {
            'name': 'ALS',
            'onset_type': 'mixed',
            'total_cycles': 30,
            'months_per_cycle': 1.5,
            'phase_boundaries': {
                'preclinical': 5, 'symptomatic': 20, 'advanced': 30
            },
            'pruning_rates': {
                'prefrontal_limbic': 0.04,
                'upper_motor': 0.10,
                'lower_motor': 0.12,
                'neuromuscular_junction': 0.08,
                'bulbar': 0.05,
                'respiratory': 0.06,
                'fine_motor': 0.09,
                'final_output': 0.04
            },
            'pruning_acceleration': 0.005,
            'autophagy_clearance': 0.05,
            'aggregate_accumulation': 0.8,
            'toxicity_threshold': 0.2,
            'base_regrowth': 0.02,
            'excitotoxicity_base': 0.05,
            'excitotoxicity_progression': 0.008,
            'motor_vulnerability': 2.0,
            'limbic_vulnerability': 0.8,
            'instability_gain': 3.0,
            'complement_bias': 0.7,
            'corruption_spread_radius': 5,
            'corruption_magnitude': 0.005,
            'flare_threshold': 0.3,
            'flare_probability': 0.10,
            'flare_magnitude': 2.0,
            # v3.0 Tier 2 parameters
            'nad_depletion_rate': 0.015,
            'nad_compensation_strength': 0.008,
            'nad_burnout_threshold': 0.25,
        }

    @staticmethod
    def als_bulbar_onset_profile():
        profile = DiseaseProfile.als_profile()
        profile['name'] = 'ALS_Bulbar_Onset'
        profile['onset_type'] = 'bulbar'
        profile['pruning_rates']['bulbar'] = 0.14
        profile['pruning_rates']['respiratory'] = 0.10
        profile['pruning_rates']['fine_motor'] = 0.06
        profile['pruning_rates']['lower_motor'] = 0.08
        return profile

    @staticmethod
    def als_limb_onset_profile():
        profile = DiseaseProfile.als_profile()
        profile['name'] = 'ALS_Limb_Onset'
        profile['onset_type'] = 'limb'
        profile['pruning_rates']['fine_motor'] = 0.14
        profile['pruning_rates']['lower_motor'] = 0.14
        profile['pruning_rates']['bulbar'] = 0.04
        profile['pruning_rates']['respiratory'] = 0.05
        return profile

    @staticmethod
    def mdd_profile():
        return {
            'name': 'MDD',
            'onset_type': 'none',
            'total_cycles': 30,
            'months_per_cycle': 1.0,
            'phase_boundaries': {
                'preclinical': 3, 'symptomatic': 15, 'advanced': 30
            },
            'pruning_rates': {
                'prefrontal_limbic': 0.10,
                'upper_motor': 0.02,
                'lower_motor': 0.02,
                'neuromuscular_junction': 0.01,
                'bulbar': 0.01,
                'respiratory': 0.01,
                'fine_motor': 0.01,
                'final_output': 0.02
            },
            'pruning_acceleration': 0.002,
            'autophagy_clearance': 0.30,
            'aggregate_accumulation': 0.2,
            'toxicity_threshold': 0.5,
            'base_regrowth': 0.08,
            'excitotoxicity_base': 0.03,
            'excitotoxicity_progression': 0.002,
            'motor_vulnerability': 0.5,
            'limbic_vulnerability': 2.0,
            'instability_gain': 2.0,
            'complement_bias': 0.5,
            'corruption_spread_radius': 3,
            'corruption_magnitude': 0.002,
            'flare_threshold': 0.5,
            'flare_probability': 0.05,
            'flare_magnitude': 1.5,
            # v3.0 Tier 2 — MDD: healthier mitochondria, slower NAD depletion
            'nad_depletion_rate': 0.008,
            'nad_compensation_strength': 0.012,
            'nad_burnout_threshold': 0.20,
        }

    @staticmethod
    def als_mdd_comorbid_profile():
        return {
            'name': 'ALS_MDD_Comorbid',
            'onset_type': 'mixed',
            'total_cycles': 30,
            'months_per_cycle': 1.5,
            'phase_boundaries': {
                'preclinical': 4, 'symptomatic': 18, 'advanced': 30
            },
            'pruning_rates': {
                'prefrontal_limbic': 0.08,
                'upper_motor': 0.10,
                'lower_motor': 0.12,
                'neuromuscular_junction': 0.08,
                'bulbar': 0.05,
                'respiratory': 0.06,
                'fine_motor': 0.09,
                'final_output': 0.05
            },
            'pruning_acceleration': 0.005,
            'autophagy_clearance': 0.05,
            'aggregate_accumulation': 0.8,
            'toxicity_threshold': 0.2,
            'base_regrowth': 0.02,
            'excitotoxicity_base': 0.05,
            'excitotoxicity_progression': 0.008,
            'motor_vulnerability': 2.0,
            'limbic_vulnerability': 1.8,
            'instability_gain': 3.0,
            'complement_bias': 0.7,
            'corruption_spread_radius': 5,
            'corruption_magnitude': 0.006,
            'flare_threshold': 0.25,
            'flare_probability': 0.12,
            'flare_magnitude': 2.0,
            # v3.0 Tier 2 — comorbid: slightly worse NAD dynamics
            'nad_depletion_rate': 0.018,
            'nad_compensation_strength': 0.007,
            'nad_burnout_threshold': 0.28,
        }

    @staticmethod
    def control_profile():
        return {
            'name': 'Control',
            'onset_type': 'none',
            'total_cycles': 30,
            'months_per_cycle': 1.5,
            'phase_boundaries': {
                'preclinical': 30, 'symptomatic': 30, 'advanced': 30
            },
            'pruning_rates': {
                'prefrontal_limbic': 0.01,
                'upper_motor': 0.01,
                'lower_motor': 0.01,
                'neuromuscular_junction': 0.01,
                'bulbar': 0.01,
                'respiratory': 0.01,
                'fine_motor': 0.01,
                'final_output': 0.01
            },
            'pruning_acceleration': 0.0,
            'autophagy_clearance': 0.50,
            'aggregate_accumulation': 0.1,
            'toxicity_threshold': 0.8,
            'base_regrowth': 0.15,
            'excitotoxicity_base': 0.01,
            'excitotoxicity_progression': 0.0,
            'motor_vulnerability': 1.0,
            'limbic_vulnerability': 1.0,
            'instability_gain': 1.0,
            'complement_bias': 0.5,
            'corruption_spread_radius': 2,
            'corruption_magnitude': 0.001,
            'flare_threshold': 0.8,
            'flare_probability': 0.0,
            'flare_magnitude': 1.0,
            # v3.0 Tier 2 — control: robust NAD homeostasis
            'nad_depletion_rate': 0.005,
            'nad_compensation_strength': 0.020,
            'nad_burnout_threshold': 0.15,
        }


# ============================================================
# SECTION 11: MAIN SIMULATION PIPELINE — v3.0
# ============================================================

class ProgressionPipeline:
    """
    All symptoms and therapeutic changes derived from network architecture
    state (masks, weights, sparsity, aggregate load, NAD levels,
    noise profile, accuracy, weight instability, autophagy efficiency).

    v3.0 simulation loop per cycle:
        1.  Check / administer treatment
        2.  Compute pre-pruning sparsities and noise (for NAD update)
        3.  Update NAD buffer (Tier 2) using pre-pruning state + treatment
        4.  Record NAD buffer
        5.  Stochastic flare check (per layer, sparsity-gated)
        6.  Pruning (with NAD vicious-cycle amplifier + flare boost)
        7.  Toxic aggregate weight corruption (with NAD magnitude scaling)
        8.  Autophagy clearance (dynamic Tier 3 with NAD + aggregate vicious cycle)
        9.  Regrowth (gradient-guided if treatment; NAD-penalised)
       10.  Apply masks
       11.  Training (with post-pruning sparsity-derived excitotoxicity noise)
       12.  Accumulate gradients for future guided regrowth
       13.  Evaluate accuracy and per-class
       14.  Compute symptoms from architecture state (including NAD + autophagy)
       15.  Record all metrics including month, phase, flares, NAD, autophagy
    """

    def __init__(self, disease_profile, treatment_schedule=None,
                 pretrain_epochs=50, seed=42):
        torch.manual_seed(seed)
        np.random.seed(seed)

        self.config = disease_profile
        self.treatment_schedule = treatment_schedule
        self.pretrain_epochs = pretrain_epochs
        self.seed = seed

        self.model = MotorCircuitNetwork()
        self.data = MotorTaskGenerator(n_samples=2000, seed=seed)

        self.pruning_engine = MicroglialPruningEngine(
            self.model,
            complement_bias=self.config['complement_bias']
        )

        self.aggregates = ToxicAggregateTracker(
            layer_names=MotorCircuitNetwork.LAYER_NAMES,
            clearance_rate=self.config['autophagy_clearance'],
            accumulation_rate=self.config['aggregate_accumulation'],
            toxicity_threshold=self.config['toxicity_threshold'],
            corruption_spread_radius=self.config.get(
                'corruption_spread_radius', 5
            ),
            corruption_magnitude=self.config.get(
                'corruption_magnitude', 0.005
            )
        )

        # v3.0: Tier 2 NAD buffer
        self.nad_buffer = MitochondrialNADBuffer(
            layer_names=MotorCircuitNetwork.LAYER_NAMES,
            initial_nad=1.0,
            depletion_rate=self.config.get('nad_depletion_rate', 0.015),
            compensation_strength=self.config.get('nad_compensation_strength', 0.008),
            burnout_threshold=self.config.get('nad_burnout_threshold', 0.25)
        )

        self.excitotoxicity = ExcitotoxicityEngine(
            base_noise=self.config['excitotoxicity_base'],
            progression_rate=self.config['excitotoxicity_progression'],
            motor_vulnerability=self.config['motor_vulnerability'],
            limbic_vulnerability=self.config['limbic_vulnerability'],
            instability_gain=self.config.get('instability_gain', 3.0)
        )

        self.symptom_mapper = SymptomMapper()
        self.criterion = nn.CrossEntropyLoss()
        self.base_lr = 0.001
        self.results = []

        # Patient heterogeneity — seed-based vulnerability modifiers
        rng_het = np.random.RandomState(seed + 1000)
        self.vulnerability_modifiers = {}
        for ln in MotorCircuitNetwork.LAYER_NAMES:
            self.vulnerability_modifiers[ln] = rng_het.uniform(0.85, 1.15)

    def _get_phase(self, cycle):
        boundaries = self.config.get('phase_boundaries', {})
        if cycle < boundaries.get('preclinical', 5):
            return 'preclinical'
        elif cycle < boundaries.get('symptomatic', 20):
            return 'symptomatic'
        else:
            return 'advanced'

    def _get_month(self, cycle):
        return cycle * self.config.get('months_per_cycle', 1.5)

    def pretrain(self):
        optimizer = optim.Adam(self.model.parameters(), lr=self.base_lr)
        x_train, y_train = self.data.get_train()
        x_test, y_test = self.data.get_test()

        for _ in range(self.pretrain_epochs):
            self.model.train()
            idx = torch.randperm(len(x_train))[:256]
            out = self.model(x_train[idx])
            loss = self.criterion(out, y_train[idx])
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        acc, per_class = evaluate(self.model, x_test, y_test)
        return acc, per_class

    def run(self):
        baseline_acc, baseline_per_class = self.pretrain()
        x_train, y_train = self.data.get_train()
        x_test, y_test = self.data.get_test()
        total_cycles = self.config['total_cycles']

        # ---- Baseline record ----
        baseline_record = {
            'cycle': -1,
            'month': 0.0,
            'phase': 'baseline',
            'condition': self.config['name'],
            'onset_type': self.config.get('onset_type', 'mixed'),
            'accuracy': baseline_acc,
            'train_loss': 0.0,
            'total_sparsity': 0.0,
            'dose_given': False,
            'treatment_active': False,
            'treatment_decay': 0.0,
            'treatment_type': 'none',
            'flare_active': False,
            'flare_layers': '[]'
        }
        for ln in MotorCircuitNetwork.LAYER_NAMES:
            baseline_record[f'sparsity_{ln}'] = 0.0
            baseline_record[f'aggregate_{ln}'] = 0.0
            baseline_record[f'noise_{ln}'] = 0.0
            baseline_record[f'weight_instability_{ln}'] = 0.0
            baseline_record[f'nad_{ln}'] = 1.0
        for c_id, c_acc in baseline_per_class.items():
            baseline_record[f'class_{c_id}_acc'] = c_acc

        baseline_symptoms = self.symptom_mapper.compute_symptoms(
            {ln: 0.0 for ln in MotorCircuitNetwork.LAYER_NAMES},
            baseline_acc,
            {ln: 0.0 for ln in MotorCircuitNetwork.LAYER_NAMES},
            0.0,
            {ln: 0.0 for ln in MotorCircuitNetwork.LAYER_NAMES
             if ln != 'final_output'},
            self.pruning_engine.get_alive_counts(),
            self.pruning_engine.get_total_counts(),
            False,
            weight_instabilities={ln: 0.0
                                  for ln in MotorCircuitNetwork.LAYER_NAMES},
            nad_levels={ln: 1.0 for ln in MotorCircuitNetwork.LAYER_NAMES},
            autophagy_efficiencies={ln: self.config['autophagy_clearance']
                                    for ln in MotorCircuitNetwork.LAYER_NAMES}
        )
        for sname, sval in baseline_symptoms.items():
            baseline_record[f'symptom_{sname}'] = sval
        self.results.append(baseline_record)

        # ---- Main simulation loop ----
        for cycle in range(total_cycles):
            record = {
                'cycle': cycle,
                'month': self._get_month(cycle),
                'phase': self._get_phase(cycle),
                'condition': self.config['name'],
                'onset_type': self.config.get('onset_type', 'mixed'),
            }

            # 1. Treatment
            null_effects = {
                'regrowth_boost': 0.0, 'noise_reduction': 0.0,
                'pruning_reduction': 0.0, 'plasticity_lr_multiplier': 1.0,
                'selective_regrow_targets': None,
                'active': False, 'decay_factor': 0.0,
                'treatment_type': 'none',
                'nad_boost': 0.0,
                'autophagy_boost': 1.0,
            }
            dose_given = False
            treatment_effects = null_effects

            if self.treatment_schedule is not None:
                dose_given = self.treatment_schedule.check_and_administer(cycle)
                treatment_effects = self.treatment_schedule.get_effects(cycle=cycle)

            record['dose_given'] = dose_given
            record['treatment_active'] = treatment_effects['active']
            record['treatment_decay'] = treatment_effects.get('decay_factor', 0.0)
            record['treatment_type'] = treatment_effects.get(
                'treatment_type', 'none'
            )

            # 2. Pre-pruning sparsities and noise (for NAD update)
            pre_sparsities = self.pruning_engine.get_all_sparsities()
            pre_noise = self.excitotoxicity.get_noise_profile(
                cycle, current_sparsities=pre_sparsities,
                treatment_effects=treatment_effects
            )

            # 3. Update NAD buffer (Tier 2) — energy demand from sparsity + noise
            self.nad_buffer.update(
                pre_sparsities, pre_noise,
                treatment_nad_boost=treatment_effects.get('nad_boost', 0.0)
            )

            # 4. Record NAD buffer
            self.nad_buffer.record()

            # 5. Stochastic flare check
            flare_active = {}
            flare_thresh = self.config.get('flare_threshold', 0.3)
            flare_prob = self.config.get('flare_probability', 0.10)
            for layer_name in self.config['pruning_rates']:
                sp = self.pruning_engine.get_layer_sparsity(layer_name)
                if sp > flare_thresh and np.random.random() < flare_prob:
                    flare_active[layer_name] = True
                else:
                    flare_active[layer_name] = False

            record['flare_active'] = any(flare_active.values())
            record['flare_layers'] = json.dumps(
                [ln for ln, a in flare_active.items() if a]
            )

            # 6. Pruning (with Tier 2 NAD vicious-cycle amplifier)
            newly_pruned_per_layer = {}
            for layer_name, base_rate in self.config['pruning_rates'].items():
                het_mod = self.vulnerability_modifiers.get(layer_name, 1.0)
                acceleration = self.config['pruning_acceleration'] * cycle
                effective_rate = base_rate * het_mod + acceleration
                effective_rate *= max(
                    0.0, 1.0 - treatment_effects['pruning_reduction']
                )

                # Flare burst
                if flare_active.get(layer_name, False):
                    effective_rate *= self.config.get('flare_magnitude', 2.0)

                # v3.0: NAD vicious-cycle amplifier
                # Low NAD -> nad_mult < 1.0 -> (2.0 - nad_mult) > 1.0 -> more pruning
                nad_mult = self.nad_buffer.get_nad_multiplier(layer_name)
                effective_rate *= (2.0 - nad_mult)

                tox_mult = self.aggregates.get_toxicity_multiplier(layer_name)
                newly_pruned_frac, pruned_indices = (
                    self.pruning_engine.prune_layer(
                        layer_name, effective_rate, tox_mult
                    )
                )
                self.aggregates.accumulate(layer_name, newly_pruned_frac)
                newly_pruned_per_layer[layer_name] = pruned_indices

            # 7. Toxic aggregate weight corruption (v3.0: NAD-scaled magnitude)
            self.aggregates.corrupt_surviving_weights(
                self.model, self.pruning_engine.masks,
                newly_pruned_per_layer, nad_buffer=self.nad_buffer
            )

            # 8. Autophagy clearance (v3.0: dynamic Tier 3 with vicious cycle)
            agg_state_for_clearance = self.aggregates.get_state()
            autophagy_base = (self.config['autophagy_clearance']
                              * treatment_effects.get('autophagy_boost', 1.0))
            self.aggregates.attempt_clearance(
                autophagy_efficiency=autophagy_base,
                nad_buffer=self.nad_buffer,
                aggregates=agg_state_for_clearance
            )

            # 9. Regrowth (v3.0: NAD-penalised regrowth efficiency)
            for layer_name in self.config['pruning_rates']:
                regrow_frac = (self.config['base_regrowth']
                               + treatment_effects['regrowth_boost'])
                # Regrowth penalty: aggregates block + NAD scales efficiency
                agg_penalty = self.aggregates.get_regrowth_penalty(layer_name)
                nad_regrowth_mult = self.nad_buffer.get_nad_multiplier(layer_name)
                regrowth_penalty = agg_penalty * nad_regrowth_mult

                if (treatment_effects.get('selective_regrow_targets')
                        == 'high_gradient_pruned'):
                    self.pruning_engine.gradient_guided_regrow(
                        layer_name, regrow_frac, regrowth_penalty
                    )
                else:
                    self.pruning_engine.regrow_layer(
                        layer_name, regrow_frac, regrowth_penalty
                    )

            # 10. Apply masks
            self.pruning_engine.apply_all_masks()

            # 11. Training with post-pruning sparsity-derived excitotoxicity
            lr = self.base_lr * treatment_effects['plasticity_lr_multiplier']
            optimizer = optim.SGD(self.model.parameters(), lr=lr)

            post_sparsities = self.pruning_engine.get_all_sparsities()
            noise_profile = self.excitotoxicity.get_noise_profile(
                cycle,
                current_sparsities=post_sparsities,
                treatment_effects=treatment_effects
            )

            train_loss = train_epoch(
                self.model, x_train, y_train, optimizer, self.criterion,
                self.pruning_engine, noise_profile=noise_profile, n_steps=3
            )

            # 12. Accumulate gradients for future guided regrowth
            self.pruning_engine.accumulate_gradients()

            # 13. Evaluate
            acc, per_class = evaluate(
                self.model, x_test, y_test, noise_profile=noise_profile
            )

            # 14. Record architecture state and compute symptoms
            sparsities = self.pruning_engine.get_all_sparsities()
            total_sparsity = self.pruning_engine.get_total_sparsity()
            self.pruning_engine.record_sparsities()
            agg_state = self.aggregates.get_state()
            self.aggregates.record()
            alive_counts = self.pruning_engine.get_alive_counts()
            total_counts = self.pruning_engine.get_total_counts()
            weight_instabilities = self.pruning_engine.get_weight_instability()
            nad_levels = {ln: self.nad_buffer.nad_levels[ln]
                          for ln in MotorCircuitNetwork.LAYER_NAMES}
            clearance_state = self.aggregates.get_clearance_state()

            symptoms = self.symptom_mapper.compute_symptoms(
                sparsities, acc, agg_state, total_sparsity,
                noise_profile, alive_counts, total_counts,
                treatment_effects['active'],
                weight_instabilities=weight_instabilities,
                nad_levels=nad_levels,
                autophagy_efficiencies=clearance_state
            )

            # 15. Pack record
            record['accuracy'] = acc
            record['train_loss'] = train_loss
            record['total_sparsity'] = total_sparsity

            for ln, sp in sparsities.items():
                record[f'sparsity_{ln}'] = sp
            for ln, ag in agg_state.items():
                record[f'aggregate_{ln}'] = ag
            for sname, sval in symptoms.items():
                record[f'symptom_{sname}'] = sval
            for c_id, c_acc in per_class.items():
                record[f'class_{c_id}_acc'] = c_acc
            for ln, nv in noise_profile.items():
                record[f'noise_{ln}'] = nv
            for ln, wi in weight_instabilities.items():
                record[f'weight_instability_{ln}'] = wi
            # v3.0: NAD levels per layer
            for ln, nad_val in nad_levels.items():
                record[f'nad_{ln}'] = nad_val
            # v3.0: effective autophagy per layer
            for ln, eff_val in clearance_state.items():
                record[f'autophagy_eff_{ln}'] = eff_val

            self.results.append(record)

            if self.treatment_schedule is not None:
                self.treatment_schedule.tick()

        return self.results

    def export_csv(self, filepath):
        if not self.results:
            return
        keys = list(self.results[0].keys())
        for r in self.results[1:]:
            for k in r.keys():
                if k not in keys:
                    keys.append(k)
        with open(filepath, 'w', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=keys, extrasaction='ignore')
            writer.writeheader()
            writer.writerows(self.results)

    def export_json(self, filepath):
        with open(filepath, 'w') as f:
            json.dump(self.results, f, indent=2, default=str)


# ============================================================
# SECTION 12: SCENARIO RUNNER — UPDATED v3.0
# ============================================================

def build_treatment_schedule(arm_name, dose_cycles, params=None):
    if dose_cycles is None:
        return None
    defaults = {
        'treatment_type': 'ketamine',
        'dose': 1.0, 'half_life': 3,
        'regrowth_strength': 0.6, 'noise_reduction': 0.5,
        'pruning_reduction': 0.4, 'plasticity_boost': 2.0
    }
    if params is not None:
        defaults.update(params)
    return TreatmentSchedule(dose_cycles=dose_cycles, **defaults)


def run_all_scenarios(seeds=None):
    if seeds is None:
        seeds = [42]

    profiles = {
        'ALS': DiseaseProfile.als_profile(),
        'ALS_Bulbar': DiseaseProfile.als_bulbar_onset_profile(),
        'ALS_Limb': DiseaseProfile.als_limb_onset_profile(),
        'MDD': DiseaseProfile.mdd_profile(),
        'ALS_MDD': DiseaseProfile.als_mdd_comorbid_profile(),
        'Control': DiseaseProfile.control_profile()
    }

    treatment_arms = {
        'no_treatment': (None, None),
        'ketamine_q5': ([5, 10, 15, 20], None),
        'ketamine_early': ([2, 4, 6], None),
        'ketamine_late': ([15, 18, 21, 24], None),
        'ketamine_maintenance': (list(range(3, 28, 3)), None),
        'riluzole_continuous': (list(range(0, 30)), {
            'treatment_type': 'riluzole', 'half_life': 2,
            'noise_reduction': 0.3, 'pruning_reduction': 0.15,
            'regrowth_strength': 0.0, 'plasticity_boost': 1.0
        }),
        'riluzole_early': (list(range(0, 15)), {
            'treatment_type': 'riluzole', 'half_life': 2,
            'noise_reduction': 0.3, 'pruning_reduction': 0.15,
            'regrowth_strength': 0.0, 'plasticity_boost': 1.0
        }),
        # v3.0: aCGR treatment arms
        'aCGR_early_pulsed': (list(range(2, 30, 4)), {
            'treatment_type': 'aCGR',
            'pulse_cycles': list(range(3, 28, 6)),
            'half_life': 4,
        }),
        'aCGR_maintenance': (list(range(3, 28, 3)), {
            'treatment_type': 'aCGR',
            'half_life': 4,
        }),
        'aCGR_late': (list(range(15, 30, 3)), {
            'treatment_type': 'aCGR',
            'pulse_cycles': list(range(16, 29, 4)),
            'half_life': 4,
        }),
    }

    all_results = {}

    for cond_name, profile in profiles.items():
        for arm_name, (dose_cycles, params) in treatment_arms.items():
            for seed in seeds:
                scenario_key = f"{cond_name}__{arm_name}__seed{seed}"
                schedule = build_treatment_schedule(
                    arm_name, dose_cycles, params
                )

                pipeline = ProgressionPipeline(
                    disease_profile=profile,
                    treatment_schedule=schedule,
                    pretrain_epochs=50,
                    seed=seed
                )

                results = pipeline.run()
                all_results[scenario_key] = results

    return all_results


# ============================================================
# SECTION 13: CONSOLE OUTPUT — UPDATED v3.0
# ============================================================

def print_scenario_table(scenario_name, results):
    print(f"\n{'=' * 160}")
    print(f"  {scenario_name}")
    print(f"{'=' * 160}")
    header = (
        f"{'Cyc':>3} {'Mo':>5} {'Phs':>5} | {'Acc%':>6} | {'Sprs%':>6} | "
        f"{'PL':>5} {'UM':>5} {'LM':>5} {'NMJ':>5} "
        f"{'Bul':>5} {'Rsp':>5} {'FM':>5} | "
        f"{'Dep':>5} {'Weak':>5} {'BulS':>5} {'Resp':>5} | "
        f"{'ALS':>5} {'NFL':>6} {'Cog':>5} | "
        f"{'NAD%':>5} {'Auto':>5} | {'Tx':>4} {'Flr':>3}"
    )
    print(header)
    print('-' * 160)

    for r in results:
        tx_str = ''
        if r.get('dose_given', False):
            tx_type = r.get('treatment_type', '?')
            tx_str = tx_type[:3].upper()
        elif r.get('treatment_active', False):
            tx_str = f"{r.get('treatment_decay', 0.0):.2f}"

        flr_str = '*' if r.get('flare_active', False) else ''

        cyc = r.get('cycle', -1)
        mo = r.get('month', 0.0)
        phs = r.get('phase', '?')[:5]
        acc = r.get('accuracy', 0.0)
        sp = r.get('total_sparsity', 0.0) * 100.0
        pl = r.get('sparsity_prefrontal_limbic', 0.0) * 100.0
        um = r.get('sparsity_upper_motor', 0.0) * 100.0
        lm = r.get('sparsity_lower_motor', 0.0) * 100.0
        nmj = r.get('sparsity_neuromuscular_junction', 0.0) * 100.0
        bul = r.get('sparsity_bulbar', 0.0) * 100.0
        rsp = r.get('sparsity_respiratory', 0.0) * 100.0
        fm = r.get('sparsity_fine_motor', 0.0) * 100.0
        dep = r.get('symptom_depression_score', 0.0)
        wk = r.get('symptom_weakness_score', 0.0)
        buls = r.get('symptom_bulbar_score', 0.0)
        resp = r.get('symptom_respiratory_risk_pct', 0.0)
        als = r.get('symptom_alsfrs_r_proxy', 0.0)
        nfl = r.get('symptom_biomarker_nfl_proxy', 0.0)
        cog = r.get('symptom_cognitive_score', 0.0)
        nad_dep = r.get('symptom_nad_depletion_score', 0.0)
        auto_eff = r.get('symptom_autophagy_efficiency_proxy', 0.0)

        print(
            f"{cyc:3d} {mo:5.1f} {phs:>5} | {acc:6.1f} | {sp:5.1f}% | "
            f"{pl:4.1f}% {um:4.1f}% {lm:4.1f}% {nmj:4.1f}% "
            f"{bul:4.1f}% {rsp:4.1f}% {fm:4.1f}% | "
            f"{dep:5.1f} {wk:5.2f} {buls:5.2f} {resp:5.1f} | "
            f"{als:5.1f} {nfl:6.1f} {cog:5.1f} | "
            f"{nad_dep:5.1f} {auto_eff:5.1f} | {tx_str:>4} {flr_str:>3}"
        )


def print_aggregate_comparison(all_results, cycle_index=-1):
    """Print final-cycle comparison across all scenarios."""
    print(f"\n{'=' * 180}")
    print("  CROSS-SCENARIO COMPARISON (final cycle)")
    print(f"{'=' * 180}")
    header = (
        f"{'Scenario':<50} | {'Acc%':>6} | {'Sprs%':>6} | "
        f"{'Dep':>5} | {'Weak':>5} | {'BulS':>5} | {'FMS':>5} | "
        f"{'ALSFRS':>6} | {'NFL':>6} | "
        f"{'Resp%':>5} | {'MotInt%':>7} | {'Cog':>5} | "
        f"{'NAD%':>5} | {'Auto%':>5}"
    )
    print(header)
    print('-' * 180)

    for scenario_name in sorted(all_results.keys()):
        results = all_results[scenario_name]
        r = results[cycle_index]
        acc = r.get('accuracy', 0.0)
        sp = r.get('total_sparsity', 0.0) * 100.0
        dep = r.get('symptom_depression_score', 0.0)
        wk = r.get('symptom_weakness_score', 0.0)
        buls = r.get('symptom_bulbar_score', 0.0)
        fms = r.get('symptom_fine_motor_score', 0.0)
        als = r.get('symptom_alsfrs_r_proxy', 0.0)
        nfl = r.get('symptom_biomarker_nfl_proxy', 0.0)
        resp = r.get('symptom_respiratory_risk_pct', 0.0)
        mi = r.get('symptom_motor_integrity_pct', 0.0)
        cog = r.get('symptom_cognitive_score', 0.0)
        nad_dep = r.get('symptom_nad_depletion_score', 0.0)
        auto_eff = r.get('symptom_autophagy_efficiency_proxy', 0.0)

        print(
            f"{scenario_name:<50} | {acc:6.1f} | {sp:5.1f}% | "
            f"{dep:5.1f} | {wk:5.2f} | {buls:5.2f} | {fms:5.2f} | "
            f"{als:6.1f} | {nfl:6.1f} | "
            f"{resp:5.1f} | {mi:6.1f}% | {cog:5.1f} | "
            f"{nad_dep:5.1f} | {auto_eff:5.1f}"
        )


def print_tier_summary(all_results, scenario_name, cycle_indices=None):
    """Print a focused Tier 1-4 state summary for a single scenario."""
    if scenario_name not in all_results:
        print(f"Scenario '{scenario_name}' not found.")
        return

    results = all_results[scenario_name]
    if cycle_indices is None:
        cycle_indices = [0, len(results) // 4, len(results) // 2,
                         3 * len(results) // 4, -1]

    print(f"\n{'=' * 120}")
    print(f"  TIER STATE SUMMARY: {scenario_name}")
    print(f"{'=' * 120}")
    header = (
        f"{'Cyc':>3} {'Mo':>5} | "
        f"{'T1:Sprs%':>8} {'T1:Prune':>8} | "
        f"{'T2:NAD%':>7} {'T2:Dep%':>7} | "
        f"{'T3:Auto%':>8} {'T3:Agg':>6} | "
        f"{'T4:Acc%':>7} {'T4:ALS':>6} {'T4:NFL':>6} | "
        f"{'Tx':>6}"
    )
    print(header)
    print('-' * 120)

    for ci in cycle_indices:
        r = results[ci]
        cyc = r.get('cycle', -1)
        mo = r.get('month', 0.0)

        # Tier 1: pruning state
        sp = r.get('total_sparsity', 0.0) * 100.0
        motor_layers = ['upper_motor', 'lower_motor', 'neuromuscular_junction',
                        'bulbar', 'respiratory', 'fine_motor']
        motor_sp = np.mean([r.get(f'sparsity_{ln}', 0.0) for ln in motor_layers]) * 100.0

        # Tier 2: NAD state
        nad_vals = [r.get(f'nad_{ln}', 1.0) for ln in MotorCircuitNetwork.LAYER_NAMES]
        avg_nad = np.mean(nad_vals) * 100.0
        nad_dep = r.get('symptom_nad_depletion_score', 0.0)

        # Tier 3: autophagy state
        auto_eff = r.get('symptom_autophagy_efficiency_proxy', 100.0)
        agg_vals = [r.get(f'aggregate_{ln}', 0.0) for ln in MotorCircuitNetwork.LAYER_NAMES]
        avg_agg = np.mean(agg_vals)

        # Tier 4: decompensation
        acc = r.get('accuracy', 0.0)
        als = r.get('symptom_alsfrs_r_proxy', 0.0)
        nfl = r.get('symptom_biomarker_nfl_proxy', 0.0)

        tx = r.get('treatment_type', 'none')[:6]

        print(
            f"{cyc:3d} {mo:5.1f} | "
            f"{sp:7.1f}% {motor_sp:7.1f}% | "
            f"{avg_nad:6.1f}% {nad_dep:6.1f}% | "
            f"{auto_eff:7.1f}% {avg_agg:5.3f} | "
            f"{acc:6.1f}% {als:5.1f} {nfl:6.1f} | "
            f"{tx:>6}"
        )


# ============================================================
# SECTION 14: ENTRY POINT — UPDATED v3.0
# ============================================================

if __name__ == '__main__':
    print("=" * 70)
    print("  ALS PROGRESSION MODEL PIPELINE v3.0")
    print("  Four-Tier Microglial Pruning-NAD+-Autophagy Continuum")
    print("  All symptoms derived from network architecture")
    print("")
    print("  Tier 1: Microglial pruning (complement + activity-dependent)")
    print("  Tier 2: Mitochondrial NAD+ buffer (depletion/compensation)")
    print("  Tier 3: Dynamic autophagy collapse (aggregate + NAD vicious cycle)")
    print("  Tier 4: Network decompensation (emergent from Tiers 1-3)")
    print("")
    print("  Treatments:")
    print("    - Ketamine (NMDA blockade + synaptogenesis)")
    print("    - Riluzole (glutamate modulation comparator)")
    print("    - aCGR (multi-tier: DXM+piracetam + NMN + NAC + senolytics)")
    print("")
    print("  Features (v2.0 preserved):")
    print("    - Sub-layer granularity (bulbar/respiratory/fine_motor)")
    print("    - Aggregate toxicity as weight corruption (TDP-43)")
    print("    - Gradient-guided selective regrowth (BDNF)")
    print("    - Sparsity-driven excitotoxicity")
    print("    - Stochastic flares & onset subtypes")
    print("    - Weight instability NFL proxy")
    print("    - Patient heterogeneity via seed modifiers")
    print("")
    print("  New in v3.0:")
    print("    - NAD+ buffer with per-layer depletion/compensation")
    print("    - Dynamic autophagy collapse (mTOR/aggregate model)")
    print("    - Vicious-cycle feedback (NAD<->autophagy<->pruning)")
    print("    - aCGR treatment (Paper 4 multi-tier regimen)")
    print("    - NAD depletion score + autophagy efficiency proxy")
    print("    - NFL biomarker scales with NAD penalty")
    print("=" * 70)

    all_results = run_all_scenarios(seeds=[42])

    # Display selected scenarios with full per-cycle tables
    display_scenarios = [
        'ALS__no_treatment__seed42',
        'ALS__ketamine_q5__seed42',
        'ALS__ketamine_maintenance__seed42',
        'ALS__riluzole_continuous__seed42',
        'ALS__aCGR_early_pulsed__seed42',
        'ALS__aCGR_maintenance__seed42',
        'ALS__aCGR_late__seed42',
        'ALS_Bulbar__no_treatment__seed42',
        'ALS_Bulbar__aCGR_early_pulsed__seed42',
        'ALS_Limb__no_treatment__seed42',
        'ALS_Limb__aCGR_early_pulsed__seed42',
        'ALS_MDD__no_treatment__seed42',
        'ALS_MDD__aCGR_early_pulsed__seed42',
        'ALS_MDD__ketamine_q5__seed42',
        'MDD__no_treatment__seed42',
        'MDD__ketamine_q5__seed42',
        'MDD__aCGR_maintenance__seed42',
        'Control__no_treatment__seed42',
    ]

    for scenario in display_scenarios:
        if scenario in all_results:
            print_scenario_table(scenario, all_results[scenario])

    # Tier state summaries for key comparisons
    tier_summary_scenarios = [
        'ALS__no_treatment__seed42',
        'ALS__aCGR_early_pulsed__seed42',
        'ALS__aCGR_late__seed42',
        'ALS__riluzole_continuous__seed42',
        'ALS_MDD__no_treatment__seed42',
        'ALS_MDD__aCGR_early_pulsed__seed42',
    ]
    for scenario in tier_summary_scenarios:
        if scenario in all_results:
            print_tier_summary(all_results, scenario)

    # Cross-scenario comparison
    print_aggregate_comparison(all_results)

    print(f"\nTotal scenarios executed: {len(all_results)}")
    print("Pipeline v3.0 complete.")

  ALS PROGRESSION MODEL PIPELINE v3.0
  Four-Tier Microglial Pruning-NAD+-Autophagy Continuum
  All symptoms derived from network architecture

  Tier 1: Microglial pruning (complement + activity-dependent)
  Tier 2: Mitochondrial NAD+ buffer (depletion/compensation)
  Tier 3: Dynamic autophagy collapse (aggregate + NAD vicious cycle)
  Tier 4: Network decompensation (emergent from Tiers 1-3)

  Treatments:
    - Ketamine (NMDA blockade + synaptogenesis)
    - Riluzole (glutamate modulation comparator)
    - aCGR (multi-tier: DXM+piracetam + NMN + NAC + senolytics)

  Features (v2.0 preserved):
    - Sub-layer granularity (bulbar/respiratory/fine_motor)
    - Aggregate toxicity as weight corruption (TDP-43)
    - Gradient-guided selective regrowth (BDNF)
    - Sparsity-driven excitotoxicity
    - Stochastic flares & onset subtypes
    - Weight instability NFL proxy
    - Patient heterogeneity via seed modifiers

  New in v3.0:
    - NAD+ buffer with per-layer depletion/compensation
   

# The End